# Test CosyVoice2 vs SparkTTS — Benchmark Tiếng Việt
**GPU:** T4 (free) | **Models:** CosyVoice2-0.5B vs Vi-SparkTTS-0.5B

Chạy từ trên xuống. So sánh:
- Chất lượng tiếng Việt (CosyVoice2 chưa train VN chính thức)
- Latency
- Voice cloning quality

In [ ]:
# Bước 0: Cài thư viện
!pip install -q soundfile num2words numpy
!pip install -q huggingface_hub

# Clone CosyVoice repo
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git 2>/dev/null || echo 'Already cloned'
!cd CosyVoice && git submodule update --init --recursive 2>/dev/null
!cd CosyVoice && pip install -q -r requirements.txt

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Bước 1: Load CosyVoice2 model
import sys, time
sys.path.insert(0, 'CosyVoice')
sys.path.insert(0, 'CosyVoice/third_party/Matcha-TTS')

t0 = time.time()
print("Downloading CosyVoice2-0.5B...")

from cosyvoice.cli.cosyvoice import AutoModel
from huggingface_hub import snapshot_download

# Download model
model_dir = snapshot_download('FunAudioLLM/CosyVoice2-0.5B',
                              local_dir='pretrained_models/CosyVoice2-0.5B')
print(f"Downloaded in {time.time()-t0:.1f}s")

# Load
t0 = time.time()
cosyvoice = AutoModel(model_dir=model_dir)
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"Sample rate: {cosyvoice.sample_rate}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Warm-up
print("Warming up...")
for i, j in enumerate(cosyvoice.inference_cross_lingual(
    '. Hello world.',
    'CosyVoice/asset/zero_shot_prompt.wav'
)):
    pass
torch.cuda.empty_cache()
print("Ready!")

In [ ]:
# Bước 2: TEST TIẾNG VIỆT — CosyVoice2
# CosyVoice2 chưa chính thức hỗ trợ VN — test xem thế nào
import time, numpy as np, soundfile as sf
from IPython.display import Audio, display

# Thủ thuật: dấu . đầu chunk giúp model đọc ổn hơn
test_sentences = [
    ". Dạ vâng, em hiểu ạ.",
    ". Dạ, xin chào anh chị. Em là Minh Anh bên Sunshine Hospitality ạ.",
    ". Cuối tuần này bên em có chương trình tham quan nghỉ dưỡng tại Phú Quốc, hoàn toàn miễn phí ạ.",
]

# Dùng default prompt (bundled trong repo)
prompt_wav = 'CosyVoice/asset/zero_shot_prompt.wav'
prompt_text = '希望你以后能够做的比我还好呦。'  # Default prompt text (Chinese)

print("=" * 60)
print("BENCHMARK: CosyVoice2-0.5B — Tiếng Việt")
print("=" * 60)

for i, text in enumerate(test_sentences):
    print(f"\n--- Test {i+1}: \"{text}\" ({len(text)} chars) ---")
    
    t0 = time.time()
    
    audio_out = None
    for _, j in enumerate(cosyvoice.inference_zero_shot(
        text,
        prompt_text,
        prompt_wav
    )):
        audio_out = j['tts_speech'].squeeze().cpu().numpy()
    
    elapsed = time.time() - t0
    sr = cosyvoice.sample_rate
    duration = len(audio_out) / sr
    
    print(f"   Latency: {elapsed:.2f}s")
    print(f"   Audio: {duration:.2f}s")
    print(f"   RTF: {elapsed/duration:.2f}x")
    
    display(Audio(audio_out, rate=sr))
    
    # Save for comparison
    sf.write(f'cosyvoice_test_{i+1}.wav', audio_out, sr)
    
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("Nghe xong so sánh với SparkTTS bên dưới")
print("=" * 60)

In [ ]:
# Bước 3: TEST CROSS-LINGUAL (Vietnamese text, default voice)
# Cross-lingual mode không cần prompt text match — tốt hơn cho VN

print("=" * 60)
print("TEST CROSS-LINGUAL MODE (có thể tốt hơn cho VN)")
print("=" * 60)

for i, text in enumerate(test_sentences):
    print(f"\n--- Cross-lingual {i+1}: \"{text}\" ---")
    
    t0 = time.time()
    
    audio_out = None
    for _, j in enumerate(cosyvoice.inference_cross_lingual(
        text,
        prompt_wav
    )):
        audio_out = j['tts_speech'].squeeze().cpu().numpy()
    
    elapsed = time.time() - t0
    duration = len(audio_out) / cosyvoice.sample_rate
    
    print(f"   Latency: {elapsed:.2f}s | Audio: {duration:.2f}s | RTF: {elapsed/duration:.2f}x")
    display(Audio(audio_out, rate=cosyvoice.sample_rate))
    
    sf.write(f'cosyvoice_crosslingual_{i+1}.wav', audio_out, cosyvoice.sample_rate)
    torch.cuda.empty_cache()

In [ ]:
# Bước 4 (tuỳ chọn): Test với voice clone giọng Minh
# Upload consent_audio.wav lên Colab trước

# from google.colab import files
# uploaded = files.upload()  # Upload consent_audio.wav
# minh_prompt = list(uploaded.keys())[0]
#
# # Vietnamese prompt text (mô tả giọng mẫu)
# minh_prompt_text = "Tôi là chủ sở hữu giọng nói này."
#
# for i, text in enumerate(test_sentences):
#     print(f"\n--- Voice Clone {i+1}: \"{text}\" ---")
#     t0 = time.time()
#     for _, j in enumerate(cosyvoice.inference_zero_shot(
#         text, minh_prompt_text, minh_prompt
#     )):
#         audio_out = j['tts_speech'].squeeze().cpu().numpy()
#     elapsed = time.time() - t0
#     duration = len(audio_out) / cosyvoice.sample_rate
#     print(f"   Latency: {elapsed:.2f}s | Audio: {duration:.2f}s")
#     display(Audio(audio_out, rate=cosyvoice.sample_rate))
#     sf.write(f'cosyvoice_minh_{i+1}.wav', audio_out, cosyvoice.sample_rate)
#     torch.cuda.empty_cache()

In [ ]:
# Bước 5: Load SparkTTS để so sánh (cùng GPU session)
# Nếu thiếu VRAM, restart runtime và chạy riêng

# # Giải phóng CosyVoice
# del cosyvoice
# torch.cuda.empty_cache()
# import gc; gc.collect()
# print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
#
# # Load SparkTTS
# !pip install -q transformers accelerate einops einx safetensors
# from transformers import AutoProcessor, AutoModel as HFAutoModel
#
# MODEL_ID = "DragonLineageAI/Vi-SparkTTS-0.5B"
# processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
# spark_model = HFAutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, device_map="cuda").eval()
# processor.model = spark_model
#
# for i, text in enumerate(test_sentences):
#     print(f"\n--- SparkTTS {i+1}: \"{text}\" ---")
#     t0 = time.time()
#     inputs = processor(text=text, return_tensors="pt")
#     inputs = {k: (v.to('cuda') if hasattr(v, 'to') else v) for k, v in inputs.items()}
#     global_tokens = inputs.pop('global_token_ids_prompt', None)
#     if global_tokens is not None:
#         global_tokens = global_tokens.to('cuda')
#     with torch.no_grad():
#         output_ids = spark_model.generate(**inputs, max_new_tokens=1200,
#                                           do_sample=True, temperature=0.7, top_k=50, top_p=0.8)
#     audio_dict = processor.decode(generated_ids=output_ids.to('cuda'),
#                                   global_token_ids_prompt=global_tokens,
#                                   input_ids_len=inputs['input_ids'].shape[-1])
#     audio = np.asarray(audio_dict['audio'], dtype=np.float32)
#     sr = int(audio_dict.get('sampling_rate', 24000))
#     elapsed = time.time() - t0
#     duration = len(audio) / sr
#     print(f"   Latency: {elapsed:.2f}s | Audio: {duration:.2f}s | RTF: {elapsed/duration:.2f}x")
#     display(Audio(audio, rate=sr))
#     sf.write(f'spark_test_{i+1}.wav', audio, sr)
#     torch.cuda.empty_cache()